# Repeater-Chain Surrogate Evaluation

Compare 5,000 paired `(time, fidelity)` samples from the Transformer-MDN surrogate with one held-out SeQUeNCe simulation.

The evaluation includes:

- time and fidelity Wasserstein distances;
- time and fidelity two-sample KS statistics;
- standardized joint energy distance for the paired distribution;
- time-fidelity correlation;
- sample-generation runtime;
- marginal and joint visualizations.

Lower is better for Wasserstein, KS, and energy distance. The correlation difference should be close to zero.

In [ ]:
from pathlib import Path
from time import perf_counter

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.spatial.distance import cdist, pdist
from scipy.stats import ks_2samp, wasserstein_distance

from train_surrogate_chain import FIDELITY_PATH, GLOBAL_FEATURE_NAMES, LINK_DISTANCES_NAME, TIME_PATH, USER_OPTIONAL_PATH
from train_surrogate_chain import load_model, resolve_device, sample_from_model

MODEL_PATH = None
MODEL_DIRECTORY = Path("models")
TEST_DIRECTORY = Path("chain/testing/h5")
TEST_FILE_INDEX = 0
SAMPLE_COUNT = 5000
JOINT_METRIC_SAMPLE_COUNT = 2000
SEED = 7

In [ ]:
if MODEL_PATH is None:
    model_paths = sorted(MODEL_DIRECTORY.glob("chain_transformer_mdn*.pt"))
    if not model_paths:
        raise FileNotFoundError(f"No chain model files found under {MODEL_DIRECTORY}")
    MODEL_PATH = max(model_paths, key=lambda path: path.stat().st_mtime)

test_paths = sorted(TEST_DIRECTORY.glob("*.h5"))
if not test_paths:
    raise FileNotFoundError(f"No HDF5 files found under {TEST_DIRECTORY}")
if not 0 <= TEST_FILE_INDEX < len(test_paths):
    raise IndexError(f"TEST_FILE_INDEX must be between 0 and {len(test_paths) - 1}")

test_path = test_paths[TEST_FILE_INDEX]
device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
artifact = load_model(MODEL_PATH, device)

print(f"Model: {MODEL_PATH}")
print(f"Test file: {test_path}")
print(f"Device: {device}")

## Load one SeQUeNCe condition and generate surrogate samples

The HDF5 attributes provide the variable-length link-distance sequence and the eight homogeneous global features. Time and fidelity are loaded as paired samples.

In [ ]:
with h5py.File(test_path, "r") as h5_file:
    attributes = h5_file[USER_OPTIONAL_PATH].attrs
    config = {LINK_DISTANCES_NAME: np.asarray(attributes[LINK_DISTANCES_NAME], dtype=np.float32).tolist()}
    for name in GLOBAL_FEATURE_NAMES:
        config[name] = float(attributes[name])
    sequence_time = np.asarray(h5_file[TIME_PATH][:], dtype=np.float32)
    sequence_fidelity = np.asarray(h5_file[FIDELITY_PATH][:], dtype=np.float32)

if sequence_time.ndim != 1 or sequence_fidelity.ndim != 1 or sequence_time.shape != sequence_fidelity.shape:
    raise ValueError("SeQUeNCe time and fidelity must be paired one-dimensional arrays")
sequence_samples = np.column_stack((sequence_time, sequence_fidelity))

if device.type == "cuda":
    torch.cuda.synchronize(device)
sample_start = perf_counter()
surrogate_samples = sample_from_model(artifact, config, SAMPLE_COUNT, SEED, device)
if device.type == "cuda":
    torch.cuda.synchronize(device)
sample_elapsed_seconds = perf_counter() - sample_start

print(f"Nodes: {len(config[LINK_DISTANCES_NAME]) + 1}")
print(f"Link distances: {config[LINK_DISTANCES_NAME]}")
print(f"SeQUeNCe samples: {len(sequence_samples):,}")
print(f"Surrogate samples: {len(surrogate_samples):,}")
print(f"End-to-end surrogate sampling time: {sample_elapsed_seconds:.6f} seconds")
print(f"Surrogate sampling rate: {len(surrogate_samples) / sample_elapsed_seconds:,.0f} samples/second")

## Distribution metrics

Marginal Wasserstein and KS metrics evaluate time and fidelity separately. Joint energy distance compares the complete paired distribution after standardizing both dimensions using the SeQUeNCe scale. This prevents milliseconds from numerically dominating fidelity.

In [ ]:
def joint_energy_distance(sequence, surrogate, sample_limit, seed):
    """Calculate standardized multivariate energy distance with bounded memory use."""
    rng = np.random.default_rng(seed)
    if len(sequence) > sample_limit:
        sequence = sequence[rng.choice(len(sequence), sample_limit, replace=False)]
    if len(surrogate) > sample_limit:
        surrogate = surrogate[rng.choice(len(surrogate), sample_limit, replace=False)]

    mean = np.mean(sequence, axis=0)
    std = np.std(sequence, axis=0)
    std[std < 1e-12] = 1.0
    sequence = (sequence - mean) / std
    surrogate = (surrogate - mean) / std
    energy_squared = 2 * np.mean(cdist(sequence, surrogate)) - np.mean(pdist(sequence)) - np.mean(pdist(surrogate))
    return float(np.sqrt(max(energy_squared, 0.0)))


def time_fidelity_correlation(samples):
    """Return Pearson correlation between paired time and fidelity samples."""
    if np.std(samples[:, 0]) == 0 or np.std(samples[:, 1]) == 0:
        return float("nan")
    return float(np.corrcoef(samples[:, 0], samples[:, 1])[0, 1])


time_wasserstein = wasserstein_distance(sequence_samples[:, 0], surrogate_samples[:, 0])
fidelity_wasserstein = wasserstein_distance(sequence_samples[:, 1], surrogate_samples[:, 1])
time_ks = ks_2samp(sequence_samples[:, 0], surrogate_samples[:, 0])
fidelity_ks = ks_2samp(sequence_samples[:, 1], surrogate_samples[:, 1])
joint_energy = joint_energy_distance(sequence_samples, surrogate_samples, JOINT_METRIC_SAMPLE_COUNT, SEED)
sequence_correlation = time_fidelity_correlation(sequence_samples)
surrogate_correlation = time_fidelity_correlation(surrogate_samples)

print(f"Time Wasserstein:                 {time_wasserstein:.6f} ms")
print(f"Normalized time Wasserstein:      {time_wasserstein / np.mean(sequence_samples[:, 0]):.6f}")
print(f"Time KS statistic:                {time_ks.statistic:.6f}")
print(f"Fidelity Wasserstein:             {fidelity_wasserstein:.6f}")
print(f"Fidelity KS statistic:            {fidelity_ks.statistic:.6f}")
print(f"Standardized joint energy:        {joint_energy:.6f}")
print(f"SeQUeNCe time-fidelity corr.:      {sequence_correlation:.6f}")
print(f"Surrogate time-fidelity corr.:     {surrogate_correlation:.6f}")
print(f"Absolute correlation difference:  {abs(sequence_correlation - surrogate_correlation):.6f}")

## Summary statistics

Percentiles are reported from 10% through 90%, followed by 95% and 99%.

In [ ]:
def summarize(samples):
    """Return mean, standard deviation, and selected percentiles."""
    percentiles = list(range(10, 100, 10)) + [95, 99]
    summary = {"mean": float(np.mean(samples)), "std": float(np.std(samples))}
    for percentile in percentiles:
        summary[f"{percentile}%"] = float(np.percentile(samples, percentile))
    return summary


def print_summary(name, sequence, surrogate):
    """Print aligned SeQUeNCe and surrogate summary statistics."""
    sequence_summary = summarize(sequence)
    surrogate_summary = summarize(surrogate)
    print(name)
    print(f"{'Metric':<10}{'SeQUeNCe':>16}{'Surrogate':>16}")
    for metric in sequence_summary:
        print(f"{metric:<10}{sequence_summary[metric]:>16.6f}{surrogate_summary[metric]:>16.6f}")
    print()


print_summary("Time [ms]", sequence_samples[:, 0], surrogate_samples[:, 0])
print_summary("Fidelity", sequence_samples[:, 1], surrogate_samples[:, 1])

## Marginal distributions

In [ ]:
combined_time = np.concatenate((sequence_samples[:, 0], surrogate_samples[:, 0]))
combined_fidelity = np.concatenate((sequence_samples[:, 1], surrogate_samples[:, 1]))
time_histogram_limit = float(np.quantile(combined_time, 0.995))
fidelity_histogram_range = (float(np.min(combined_fidelity)), float(np.max(combined_fidelity)))


def ecdf(samples):
    """Return sorted values and empirical cumulative probabilities."""
    values = np.sort(samples)
    probabilities = np.arange(1, len(values) + 1) / len(values)
    return values, probabilities


figure, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
axes[0, 0].hist(sequence_samples[:, 0], bins=60, range=(0, time_histogram_limit), density=True, alpha=0.55, label="SeQUeNCe")
axes[0, 0].hist(surrogate_samples[:, 0], bins=60, range=(0, time_histogram_limit), density=True, alpha=0.55, label="Surrogate")
axes[0, 0].set(title="Time distribution", xlabel="Time [ms]", ylabel="Probability density")
axes[0, 0].legend()

axes[0, 1].hist(sequence_samples[:, 1], bins=60, range=fidelity_histogram_range, density=True, alpha=0.55, label="SeQUeNCe")
axes[0, 1].hist(surrogate_samples[:, 1], bins=60, range=fidelity_histogram_range, density=True, alpha=0.55, label="Surrogate")
axes[0, 1].set(title="Fidelity distribution", xlabel="Fidelity", ylabel="Probability density")
axes[0, 1].legend()

for samples, label in ((sequence_samples[:, 0], "SeQUeNCe"), (surrogate_samples[:, 0], "Surrogate")):
    values, probabilities = ecdf(samples)
    axes[1, 0].plot(values, probabilities, label=label)
axes[1, 0].set(title="Time empirical CDF", xlabel="Time [ms]", ylabel="Cumulative probability")
axes[1, 0].legend()

for samples, label in ((sequence_samples[:, 1], "SeQUeNCe"), (surrogate_samples[:, 1], "Surrogate")):
    values, probabilities = ecdf(samples)
    axes[1, 1].plot(values, probabilities, label=label)
axes[1, 1].set(title="Fidelity empirical CDF", xlabel="Fidelity", ylabel="Cumulative probability")
axes[1, 1].legend()
plt.show()

## Joint time-fidelity distribution

Both panels use the same axis limits. Similar orientation, clustering, and spread indicate that the full-covariance MDN has learned the dependence between time and fidelity.

In [ ]:
plot_rng = np.random.default_rng(SEED)
sequence_plot = sequence_samples[plot_rng.choice(len(sequence_samples), min(5000, len(sequence_samples)), replace=False)]
surrogate_plot = surrogate_samples[plot_rng.choice(len(surrogate_samples), min(5000, len(surrogate_samples)), replace=False)]
combined_plot = np.vstack((sequence_plot, surrogate_plot))
time_limits = (float(np.min(combined_plot[:, 0])), float(np.max(combined_plot[:, 0])))
fidelity_limits = (float(np.min(combined_plot[:, 1])), float(np.max(combined_plot[:, 1])))

figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True, constrained_layout=True)
axes[0].scatter(sequence_plot[:, 0], sequence_plot[:, 1], s=10, alpha=0.25)
axes[0].set_title(f"SeQUeNCe | correlation = {sequence_correlation:.3f}")
axes[1].scatter(surrogate_plot[:, 0], surrogate_plot[:, 1], s=10, alpha=0.25, color="tab:orange")
axes[1].set_title(f"Surrogate | correlation = {surrogate_correlation:.3f}")
for axis in axes:
    axis.set(xlabel="Time [ms]", ylabel="Fidelity", xlim=time_limits, ylim=fidelity_limits)
plt.show()

Change `TEST_FILE_INDEX` and rerun the notebook to evaluate another testing condition. Set `MODEL_PATH = Path("models/model_name.pt")` when selecting a specific trained model.